#### SUPPLY CHAIN PERFORMANCE & RISK ANALYTICS
##### DATA CLEANING & FEATURE ENGINEERING

In [1]:
# 1. Import Libraries

import pandas as pd
import numpy as np

In [2]:
# 2. Define Path
data_path = ("C:/Users/Admin/Downloads/Portfolio Projects/Supply Chain Performance and Risk Analytics/data/raw/DataCoSupplyChainDataset.csv")

In [3]:
# 3. Load Raw Dataset

print("\n1. LOAD RAW DATA", "\n")

df = pd.read_csv(data_path,encoding="latin-1")

print(f"Raw dataset loaded successfully.")
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns):,}")


1. LOAD RAW DATA 



Raw dataset loaded successfully.
Rows    : 180,519
Columns : 53


In [4]:
# 4. Create Working Copy

print("\n2. CREATE WORKING COPY", "\n")

clean_df = df.copy()

print("Working copy created.")
print("Raw dataset remains unchanged.")


2. CREATE WORKING COPY 

Working copy created.
Raw dataset remains unchanged.


In [5]:
# 5. Convert Date Columns

print("\n3. DATE STANDARDIZATION", "\n")

date_columns = ["order date (DateOrders)", "shipping date (DateOrders)"]

for column in date_columns:
    clean_df[column] = pd.to_datetime(
        clean_df[column],
        errors="coerce"
    )

print("Date columns converted to datetime.")

print(
    f"Invalid order dates : "
    f"{clean_df['order date (DateOrders)'].isna().sum():,}"
)

print(
    f"Invalid shipping dates : "
    f"{clean_df['shipping date (DateOrders)'].isna().sum():,}"
)


3. DATE STANDARDIZATION 

Date columns converted to datetime.
Invalid order dates : 0
Invalid shipping dates : 0


In [6]:
# 6. Remove Unnecessary / Sensitive Fields

print("\n4. FIELD SELECTION", "\n")

exclude_columns = [
    # Sensitive customer information
    "Customer Email", "Customer Fname", "Customer Lname", 
    "Customer Password", "Customer Street", 
    # Completely missing / non-analytical field
    "Product Description", 
    # Non-analytical image URL
    "Product Image", 
    # Extremely incomplete geographic field
    "Order Zipcode", 
    # Constant field
    "Product Status"
]

existing_excluded_columns = [
    column
    for column in exclude_columns
    if column in clean_df.columns
]

clean_df = clean_df.drop(
    columns = existing_excluded_columns
)

print("Excluded fields :")

for column in existing_excluded_columns:
    print(f"  - {column}")

print(
    f"\nColumns remaining : "
    f"{len(clean_df.columns)}"
)


4. FIELD SELECTION 

Excluded fields :
  - Customer Email
  - Customer Fname
  - Customer Lname
  - Customer Password
  - Customer Street
  - Product Description
  - Product Image
  - Order Zipcode
  - Product Status

Columns remaining : 44


In [7]:
# 7. Remove Redundant Identifier Columns

print("\n5. REDUNDANT IDENTIFIER REVIEW", "\n")

redundant_identifier_columns = ["Customer Id", "Order Item Cardprod Id", "Product Category Id"]

# These are retained through their canonical counterparts:
# Order Customer Id
# Product Card Id
# Category Id

existing_redundant_columns = [
    column
    for column in redundant_identifier_columns
    if column in clean_df.columns
]

clean_df = clean_df.drop(
    columns = existing_redundant_columns
)

print("Redundant identifier fields removed :")

for column in existing_redundant_columns:
    print(f"  - {column}")

print("\nCanonical identifier fields retained :")

print("  - Order Customer Id")
print("  - Product Card Id")
print("  - Category Id")



5. REDUNDANT IDENTIFIER REVIEW

 

Redundant identifier fields removed :
  - Customer Id
  - Order Item Cardprod Id
  - Product Category Id

Canonical identifier fields retained :
  - Order Customer Id
  - Product Card Id
  - Category Id


In [9]:
# 8. Handle Missing Values

print("\n6. MISSING VALUE HANDLING", "\n")

missing_before = clean_df.isna().sum()
missing_before = missing_before[missing_before > 0]

print("Missing values remaining after field selection :")

if missing_before.empty:
    print("  No missing values.")
else:
    print(missing_before.to_string())

# Customer Zipcode has only 3 missing values.
# We intentionally do NOT impute it because it is not required for the core supply-chain analysis.


6. MISSING VALUE HANDLING 

Missing values remaining after field selection :
Customer Zipcode    3


In [10]:
# 9. Create Date-Based Analytical Fields

print("\n7. DATE-BASED FEATURES", "\n")

clean_df["Order Year"] = (
    clean_df["order date (DateOrders)"].dt.year
)

clean_df["Order Month"] = (
    clean_df["order date (DateOrders)"].dt.month
)

clean_df["Order Month Name"] = (
    clean_df["order date (DateOrders)"].dt.month_name()
)

clean_df["Order Quarter"] = (
    clean_df["order date (DateOrders)"].dt.quarter
)

clean_df["Order Day"] = (
    clean_df["order date (DateOrders)"].dt.day
)

clean_df["Order Day of Week"] = (
    clean_df["order date (DateOrders)"].dt.day_name()
)

clean_df["Order Date Only"] = (
    clean_df["order date (DateOrders)"].dt.date
)

print("""Created :
  - Order Year
  - Order Month
  - Order Month Name
  - Order Quarter
  - Order Day
  - Order Day of Week
  - Order Date Only""")


7. DATE-BASED FEATURES 

Created :
  - Order Year
  - Order Month
  - Order Month Name
  - Order Quarter
  - Order Day
  - Order Day of Week
  - Order Date Only


In [11]:
# 10. Create Shipping Performance Features

print("\n8. SHIPPING PERFORMANCE FEATURES", "\n")

clean_df["Shipping Duration Variance"] = (clean_df["Days for shipping (real)"]
    - clean_df["Days for shipment (scheduled)"]
)

clean_df["Shipping Performance"] = np.select(
    [
        clean_df["Shipping Duration Variance"] < 0,
        clean_df["Shipping Duration Variance"] == 0,
        clean_df["Shipping Duration Variance"] > 0
    ],
    [
        "Faster than scheduled",
        "On schedule",
        "Slower than scheduled"
    ],
    default="Unknown"
)

clean_df["Delayed Shipment Flag"] = np.where(
    clean_df["Shipping Duration Variance"] > 0,
    1,
    0
)

clean_df["On Schedule Flag"] = np.where(
    clean_df["Shipping Duration Variance"] == 0,
    1,
    0
)

clean_df["Faster Shipment Flag"] = np.where(
    clean_df["Shipping Duration Variance"] < 0,
    1,
    0
)

print("""Created :
  - Shipping Duration Variance
  - Shipping Performance
  - Delayed Shipment Flag
  - On Schedule Flag
  - Faster Shipment Flag""")


8. SHIPPING PERFORMANCE FEATURES 

Created :
  - Shipping Duration Variance
  - Shipping Performance
  - Delayed Shipment Flag
  - On Schedule Flag
  - Faster Shipment Flag


In [12]:
# 11. Create Financial Exposure Fields

print("\n9. FINANCIAL ANALYTICAL FEATURES", "\n")

clean_df["Delayed Sales"] = np.where(
    clean_df["Delayed Shipment Flag"] == 1,
    clean_df["Sales"],
    0
)

clean_df["Delayed Profit"] = np.where(
    clean_df["Delayed Shipment Flag"] == 1,
    clean_df["Order Profit Per Order"],
    0
)

print("""Created :
  - Delayed Sales
  - Delayed Profit""")

print("\nNote : These represent sales/profit associated with delayed transactions.")

print("They are NOT interpreted as lost revenue or lost profit.")


9. FINANCIAL ANALYTICAL FEATURES 

Created :
  - Delayed Sales
  - Delayed Profit

Note : These represent sales/profit associated with delayed transactions.
They are NOT interpreted as lost revenue or lost profit.


In [14]:
# 12. Validate Cleaned Dataset

print("\n10. CLEANED DATA VALIDATION", "\n")

print(f"Rows after cleaning    : {len(clean_df):,}")

print(f"Columns after cleaning : {len(clean_df.columns):,}")

print(f"Duplicate rows         : {clean_df.duplicated().sum():,}")

print(f"Duplicate Order Item Ids : {clean_df['Order Item Id'].duplicated().sum():,}")


10. CLEANED DATA VALIDATION 

Rows after cleaning    : 180,519
Columns after cleaning : 55
Duplicate rows         : 0
Duplicate Order Item Ids : 0


In [15]:
# 13. Validate Derived Features

print("\n11. DERIVED FEATURE VALIDATION", "\n")

print("\nShipping Performance :")
print(clean_df["Shipping Performance"].value_counts().to_string())

print("\nDelayed Shipment Flag :")
print(clean_df["Delayed Shipment Flag"].value_counts().sort_index().to_string())

print("\nShipping Duration Variance :")
print(clean_df["Shipping Duration Variance"].describe().to_string())


11. DERIVED FEATURE VALIDATION 


Shipping Performance :
Shipping Performance
Slower than scheduled    103400
Faster than scheduled     43366
On schedule               33753

Delayed Shipment Flag :
Delayed Shipment Flag
0     77119
1    103400

Shipping Duration Variance :
count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000


In [17]:
# 14. Financial Feature Validation

print("\n12. FINANCIAL FEATURE VALIDATION", "\n")

print(f"Total Sales  : {clean_df['Sales'].sum():,.4f}")

print(f"Sales associated with delays  : {clean_df['Delayed Sales'].sum():,.4f}")

print(f"Total Profit  : {clean_df['Order Profit Per Order'].sum():,.4f}")

print(f"Profit associated with delays  : {clean_df['Delayed Profit'].sum():,.4f}")


12. FINANCIAL FEATURE VALIDATION 

Total Sales  : 36,784,735.0134
Sales associated with delays  : 21,025,185.1621
Total Profit  : 3,966,902.9741
Profit associated with delays  : 2,231,966.6516


In [18]:
# 15. Final Missing-Value Check

print("\n13. FINAL MISSING VALUE CHECK", "\n")

final_missing = clean_df.isna().sum()
final_missing = final_missing[final_missing > 0].sort_values(ascending=False)

if final_missing.empty :
    print("No missing values remain.")
else:
    print(final_missing.to_string())


13. FINAL MISSING VALUE CHECK 

Customer Zipcode    3


In [26]:
# 16. Save Processed Dataset

print("\n14. SAVE CLEANED DATASET", "\n")

processed_data_path = ("C:/Users/Admin/Downloads/Portfolio Projects/Supply Chain Performance and Risk Analytics/data/processed/cleaned_SupplyChainDataset.csv")
clean_df.to_csv(processed_data_path, 
    index=False,
    encoding="utf-8"
)

print(f"Cleaned dataset saved successfully :"
      f"\n {processed_data_path}")



14. SAVE CLEANED DATASET 

Cleaned dataset saved successfully :
 C:/Users/Admin/Downloads/Portfolio Projects/Supply Chain Performance and Risk Analytics/data/processed/cleaned_SupplyChainDataset.csv


In [28]:
# 17. Final Summary

print("\n" + "_" * 80)
print("\nDATA CLEANING COMPLETE")
print("_" * 80)

print(f"""
Raw records     : {len(df):,}
Cleaned records : {len(clean_df):,}
Raw columns     : {len(df.columns):,}
Cleaned columns : {len(clean_df.columns):,}

Analytical grain  : One row per order item

Processed file:
{processed_data_path}
""")


________________________________________________________________________________

DATA CLEANING COMPLETE
________________________________________________________________________________

Raw records     : 180,519
Cleaned records : 180,519
Raw columns     : 53
Cleaned columns : 55

Analytical grain  : One row per order item

Processed file:
C:/Users/Admin/Downloads/Portfolio Projects/Supply Chain Performance and Risk Analytics/data/processed/cleaned_SupplyChainDataset.csv

